<a href="https://colab.research.google.com/github/Daniel-534/ModelamientoNCuerpos/blob/main/S_Stars_Dynamics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evolución dinámica e interacciones gravitacionales del cúmulo de estrellas S en el entorno del agujero negro supermasivo Sgr A*

# Modelado N cuerpos, Mecánica Celeste

## Fuentes

### Artículos

* [An Update on Monitoring Stellar Orbits in the Galactic Center](https://arxiv.org/abs/1611.09144)
* [Kinematic Structure of the Galactic Center S-cluster](https://arxiv.org/abs/2006.11399)
* [Sagittarius A* - The Milky Way Supermassive Black Hole](https://arxiv.org/abs/2503.20081)


### Bases de datos

* [25yrs monitoring of stellar orbits in the GC (Gillessen+, 2017)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/837/30)
* [Kinematic structure of the Galactic Center S cluster (Ali+, 2020)](https://vizier.cds.unistra.fr/viz-bin/VizieR?-source=J/ApJ/896/100)

## Preguntas de investigación

1. **Dinámica Newtoniana base:** ¿Cómo evolucionan las órbitas de las estrellas S (subconjunto de 5,
   luego las 39 del catálogo) al integrar numéricamente sus estados cartesianos iniciales bajo la
   influencia dominante de Sgr A*?

2. **Perturbaciones de N-cuerpos:** ¿Qué tan significativas son las interacciones gravitacionales
   estrella–estrella frente a la atracción central de Sgr A*? Comparación entre *N* simulaciones de
   2 cuerpos (estrella + Sgr A*) y una única simulación de *N+1* cuerpos.

In [2]:
!pip install -Uq pymcel celluloid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 9.5 MB/s eta 0:00:00


In [ ]:
import pymcel as pc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go
import pandas as pd

## Condiciones Iniciales

La base de datos a emplear es un archivo .tsv que contiene información, como los elementos orbitales, de 39 estrellas del cúmulo estelar cercano a Sagitario A* (Sgr A*), el agujero negro supermasivo central de la Vía Láctea, consta de estrellas rápidas llamadas "estrellas S".

In [ ]:
# Base de datos extraída del repositorio en GitHub (rama principal)
url = "https://raw.githubusercontent.com/Daniel-534/ModelamientoNCuerpos/main/Sgrestrellas.tsv"
df  = pd.read_csv(url, sep=';')
df['Name'] = df['Name'].str.strip()
df['Disk'] = df['Disk'].str.strip()

In [18]:
df.head()

,Disk,Name,a,e_a,e,e_e,i,e_i,omega,e_omega,Omega,e_Omega,Tclos,q_Tclos,e_Tclos,Simbad,_RA,_DE
0,black,S1,22.675,0.257,0.665,0.003,121.066,0.401,109.893,0.458,352.484,0.286,2000.261,,0.001,Simbad,266.41684,-29.00787
1,black,S2,5.034,0.001,0.887,0.002,137.514,0.401,73.416,0.745,235.634,1.031,2002.390,,0.020,Simbad,266.41685,-29.00777
2,black,S8,16.637,0.182,0.768,0.022,75.057,0.573,337.931,2.120,317.075,0.630,1979.216,,0.037,Simbad,266.41696,-29.00788
3,black,S9,11.125,0.030,0.791,0.036,81.876,0.458,137.854,0.573,158.079,0.229,1972.924,,0.023,Simbad,266.41690,-29.00790
4,black,S12,11.962,0.105,0.906,0.003,33.060,0.516,311.173,0.802,236.173,1.146,1995.881,,0.001,Simbad,266.41682,-29.00772


## Conversión de elementos orbitales clásicos a estado cartesiano

Dado un conjunto $(a, e, i, \Omega, \omega)$ medidos observacionalmente, calculamos el semilatus
rectum $p = a(1-e^2)$ y usamos `pymcel.elementos_a_estado(\mu, [p, e, i, \Omega, \omega, f=0])`
para obtener el vector de estado $\mathbf{r}_0, \mathbf{v}_0$ en el periastro ($f=0$).

In [19]:
# Unidades canónicas para el sistema del centro galáctico
# UL = 1 mpc (miliparsec)  |  UT = 1 año  |  UM = 1 Msol
# G se expresa en mpc³/(Msol·yr²)

G_SI   = 6.674e-11        # m³ kg⁻¹ s⁻²
mpc_m  = 3.0857e13        # metros por miliparsec
Msol   = 1.989e30         # kg por masa solar
yr_s   = 3.1557e7         # segundos por año

G_canon = G_SI * Msol * yr_s**2 / mpc_m**3   # mpc³/(Msol·yr²)
print(f"G = {G_canon:.4e} mpc³/(Msol·yr²)")

# Parámetro gravitacional de Sgr A* (μ = G·M)
M_SgrA  = 4.3e6                   # masas solares
mu_SgrA = G_canon * M_SgrA        # mpc³/yr²
print(f"μ(Sgr A*) = {mu_SgrA:.4f} mpc³/yr²")

# pc.elementos_a_estado(mu, [p, e, i, Ω, ω, f]) → [x, y, z, vx, vy, vz]
# p en mpc → posiciones en mpc, velocidades en mpc/yr

In [ ]:
# ─── Conversión orbital: todos los objetos del catálogo ──────────────────────
# Los ángulos (i, Ω, ω) están en grados; se convierten a radianes.
# Condición inicial: periastro (f = 0).  Se excluyen órbitas no ligadas (e ≥ 1).

estados_todos = []   # [x, y, z, vx, vy, vz]  (mpc, mpc/yr)
nombres_todos = []   # etiqueta de cada estrella
disco_todos   = []   # 'black' (cúmulo interno) o 'red' (cúmulo externo)

for _, est in df.iterrows():
    e = float(str(est['e']).strip())
    if e >= 1.0:          # descartar hiperbólicas / parabólicas
        print(f"  Saltando {est['Name'].strip()}: e = {e:.3f}")
        continue

    a = float(str(est['a']).strip())        # semieje mayor (mpc)
    p = a * (1 - e**2)                      # semilatus rectum (mpc)
    i = np.radians(float(str(est['i']).strip()))
    W = np.radians(float(str(est['Omega']).strip()) % 360)   # Ω mod 360°
    w = np.radians(float(str(est['omega']).strip()) % 360)   # ω mod 360°
    f = 0.0                                                   # periastro

    estado = pc.elementos_a_estado(mu_SgrA, [p, e, i, W, w, f])
    estados_todos.append(estado)
    nombres_todos.append(est['Name'].strip())
    disco_todos.append(est['Disk'].strip())

N_estrellas = len(estados_todos)
print(f"\nEstrellas con órbitas ligadas (e < 1): {N_estrellas}")
print(f"  Disco negro (cúmulo interno): {disco_todos.count('black')}")
print(f"  Disco rojo  (cúmulo externo): {disco_todos.count('red')  }")

In [ ]:
columnas_estado = ['x', 'y', 'z', 'vx', 'vy', 'vz']
df_estados = pd.DataFrame(estados_todos, index=nombres_todos, columns=columnas_estado)
df_estados.round(5)

### Masas
El agujero negro supermasivo Sgr A* en el centro galáctico tiene una masa bien determinada de ~4.15×10^6 M☉ con precisión de ~0.3%
, medida principalmente mediante el seguimiento de órbitas estelares cercanas (Genzel, Ghez, GRAVITY). Por su parte, las estrellas S más luminosas del cúmulo interno son jóvenes de tipo espectral ~B0–B3V. Dado que las masas individuales de las estrellas S se estiman mediante su tipo espectral, y su efecto perturbador es mínimo frente a Sgr A, se asignaron masas estimadas de entre 8 y 15 masas solares basadas en la literatura (Gillessen et al., 2017).

In [ ]:
# ─── Masas estelares ──────────────────────────────────────────────────────────
# Las estrellas S son de tipo B0–B3V.  Se asignan masas en [8, 14] Msol con
# semilla fija para garantizar reproducibilidad.
np.random.seed(42)
masas_todos = np.random.uniform(8.0, 14.0, N_estrellas)   # Msol

print(f"Masas asignadas a las {N_estrellas} estrellas:")
for nombre, masa in zip(nombres_todos, masas_todos):
    print(f"  {nombre:6s}: {masa:.2f} Msol")

---
## Respuesta 1 — Dinámica Newtoniana base: subconjunto de 5 estrellas

Validamos el modelo con las 5 primeras estrellas del disco negro (S1, S2, S8, S9, S12).

**Integrador:** `pymcel.ncuerpos_solucion` → `scipy.integrate.odeint` (Adams-Moulton adaptativo).
El array de resultados tiene forma `(N_cuerpos, N_pasos, 3)`.

**Verificación:** la energía total $E = K + U$ debe conservarse (error relativo < 10⁻⁵).

In [ ]:
# ─── Sistema: Sgr A* (origen) + 5 estrellas del disco interno ────────────────
# rs tiene forma (N_cuerpos, N_pasos, 3): rs[i, t, xyz]
n_demo = 5
est_5  = estados_todos[:n_demo]
nom_5  = nombres_todos[:n_demo]
mas_5  = masas_todos[:n_demo]

sistema_5 = [dict(m=mu_SgrA, r=[0., 0., 0.], v=[0., 0., 0.])]   # Sgr A*
for estado, masa in zip(est_5, mas_5):
    x, y, z, vx, vy, vz = estado
    sistema_5.append(dict(m=G_canon * masa, r=[x, y, z], v=[vx, vy, vz]))

print(f"Sistema con {len(sistema_5)} cuerpos:")
print(f"  {'Sgr A*':8s}: μ = {mu_SgrA:.4f} mpc³/yr²")
for nombre, s, masa in zip(nom_5, sistema_5[1:], mas_5):
    r0, v0 = np.linalg.norm(s['r']), np.linalg.norm(s['v'])
    print(f"  {nombre:8s}: μ = {G_canon*masa:.2e} mpc³/yr², "
          f"|r₀| = {r0:.3f} mpc, |v₀| = {v0:.3f} mpc/yr")

# ─── Integración numérica ─────────────────────────────────────────────────────
T_sim_5   = 100
N_pasos_5 = 5000
ts_5      = np.linspace(0, T_sim_5, N_pasos_5)

print(f"\nIntegrando {T_sim_5} años con {N_pasos_5} pasos (dt = {T_sim_5/N_pasos_5:.3f} yr) ...")
rs_5, vs_5, _, _, const_5 = pc.ncuerpos_solucion(sistema_5, ts_5)
# rs_5.shape = (N_cuerpos, N_pasos, 3)
print(f"¡Integración completada!  rs_5.shape = {rs_5.shape}")

# ─── Conservación de energía ──────────────────────────────────────────────────
K_5, U_5 = const_5['K'], const_5['U']
E_5      = K_5 + U_5
err_E_5  = np.abs((E_5 - E_5[0]) / E_5[0]).max()
print(f"Error relativo máx. en energía: {err_E_5:.2e}")

fig_E5, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(ts_5, (E_5 - E_5[0]) / np.abs(E_5[0]), color='tab:red')
axs[0].set(xlabel="Tiempo (años)", ylabel="(E − E₀) / |E₀|",
           title="Conservación de energía — 5 estrellas S")
axs[0].grid(True)

axs[1].plot(ts_5, K_5, label='K (cinética)',  color='tab:orange')
axs[1].plot(ts_5, U_5, label='U (potencial)', color='tab:blue')
axs[1].plot(ts_5, E_5, label='E = K + U',     color='tab:green', lw=2)
axs[1].set(xlabel="Tiempo (años)", ylabel="Energía (mpc² yr⁻²)",
           title="Componentes de energía")
axs[1].legend(); axs[1].grid(True)
plt.tight_layout(); plt.show()

# ─── Visualización 3D con Plotly ─────────────────────────────────────────────
# rs_5[i, :, :] = trayectoria del cuerpo i; rs_5[i, -1, :] = posición final
COLORS_5 = ['gold', '#4C9BE8', '#E74C3C', '#2ECC71', '#F39C12', '#9B59B6']
nombres_5_full = ['Sgr A*'] + nom_5

fig_5 = go.Figure()
for i, label in enumerate(nombres_5_full):
    x_traj, y_traj, z_traj = rs_5[i, :, 0], rs_5[i, :, 1], rs_5[i, :, 2]
    fig_5.add_trace(go.Scatter3d(
        x=x_traj, y=y_traj, z=z_traj,
        mode='lines',
        line=dict(width=3 if i == 0 else 1.5, color=COLORS_5[i]),
        name=label,
    ))
    # posición final + etiqueta
    fig_5.add_trace(go.Scatter3d(
        x=[x_traj[-1]], y=[y_traj[-1]], z=[z_traj[-1]],
        mode='markers+text',
        marker=dict(size=6 if i > 0 else 10, color=COLORS_5[i],
                    symbol='circle', line=dict(width=1, color='white')),
        text=[label], textfont=dict(size=11, color=COLORS_5[i]),
        textposition='top center', showlegend=False,
    ))

# Anotación para S2 (estrella de referencia más importante)
fig_5.add_annotation(
    text="★ S2 · T≈16 yr · e=0.887",
    showarrow=False, xref="paper", yref="paper",
    x=0.01, y=0.97, font=dict(size=12, color='#E74C3C'),
    bgcolor='rgba(0,0,0,0.45)', borderpad=4,
)

fig_5.update_layout(
    title=f"Sgr A* + 5 estrellas S — simulación de {T_sim_5} años",
    scene=dict(xaxis_title="x (mpc)", yaxis_title="y (mpc)", zaxis_title="z (mpc)",
               aspectmode='data'),
    legend=dict(font=dict(size=11)),
    margin=dict(l=0, r=0, t=40, b=0),
)
fig_5.show()

---
## Respuesta 2 — Perturbaciones N-cuerpos vs. 2 cuerpos (subconjunto de 5 estrellas)

Para cuantificar el efecto de las interacciones estrella–estrella, comparamos dos modelos:

| Modelo | Descripción |
|--------|-------------|
| **2 cuerpos** | Cada estrella se integra **por separado** con Sgr A* fijo en el origen (órbita kepleriana pura) |
| **N+1 cuerpos** | Todas las estrellas se integran **simultáneamente**; cada par interactúa gravitacionalmente |

La desviación $\Delta r(t) = |\mathbf{r}_{N+1}(t) - \mathbf{r}_\text{Kepler}(t)|$ mide
la magnitud acumulada de las perturbaciones estelares.

In [ ]:
# ─── Trayectorias kepleranas puras (2 cuerpos) para cada una de las 5 estrellas ─
# rs_2b tiene forma (2, N_pasos, 3): cuerpo 0 = Sgr A*, cuerpo 1 = estrella
print("Calculando trayectorias de 2 cuerpos...")
rs_2b_5 = []   # lista de arrays (N_pasos, 3), una por estrella

for i, (nombre, estado, masa) in enumerate(zip(nom_5, est_5, mas_5)):
    sistema_2b = [
        dict(m=mu_SgrA,        r=[0., 0., 0.],     v=[0., 0., 0.]),
        dict(m=G_canon * masa,  r=list(estado[:3]), v=list(estado[3:])),
    ]
    rs_2b_i, _, _, _, _ = pc.ncuerpos_solucion(sistema_2b, ts_5)
    rs_2b_5.append(rs_2b_i[1, :, :])   # cuerpo 1 = estrella
    print(f"  {nombre}: ✓")

# ─── Desviación relativa Δr(t) / |r(t)| ──────────────────────────────────────
fig_comp5, axes = plt.subplots(1, 2, figsize=(14, 5))

delta_max_5 = []
for i, nombre in enumerate(nom_5):
    r_nb   = rs_5[i + 1, :, :]                               # N+1 cuerpos: cuerpo i+1
    r_2b   = rs_2b_5[i]                                      # 2 cuerpos (Kepler)
    dr     = np.linalg.norm(r_nb - r_2b, axis=1)             # |Δr| (mpc)
    dr_rel = dr / (np.linalg.norm(r_nb, axis=1) + 1e-30)     # relativo

    axes[0].plot(ts_5, dr,     label=nombre)
    axes[1].plot(ts_5, dr_rel, label=nombre)
    delta_max_5.append(dr_rel.max())

for ax, ylabel, title in zip(
        axes,
        ["Δr (mpc)", "Δr / |r|  (adimensional)"],
        ["Desviación absoluta: N-cuerpos vs. 2-cuerpos",
         "Desviación relativa: N-cuerpos vs. 2-cuerpos"]):
    ax.set(xlabel="Tiempo (años)", ylabel=ylabel, title=title)
    ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

# ─── Estimación analítica de la relación de fuerzas ──────────────────────────
# F_est / F_SgrA ≈ (N × <m_est>) / (4 × M_SgrA)  (distancias comparables)
ratio_5 = n_demo * np.mean(mas_5) / (4 * 4.3e6)
print(f"\nEstimación analítica de relación de fuerzas: {ratio_5:.2e}")
print(f"→ Las perturbaciones representan ~ {ratio_5*100:.4f}% de la fuerza de Sgr A*")
print(f"Desviación relativa máxima media: {np.mean(delta_max_5):.2e}")

---
## Escalado completo — 39 estrellas S + Sgr A*

Con las condiciones iniciales de las 39 estrellas del catálogo, integramos el sistema completo de
40 cuerpos durante 100 años.

> **Nota sobre el paso temporal:** la estrella de período más corto en el cúmulo interno es S2
> (T ≈ 16 yr) y S62 tiene e = 0.98.  Con 5 000 pasos en 100 años (dt ≈ 7 días) se logra una
> precisión energética típicamente mejor que 10⁻⁵.

In [ ]:
# ─── Sistema completo: Sgr A* + N_estrellas estrellas S ──────────────────────
sistema_39 = [dict(m=mu_SgrA, r=[0., 0., 0.], v=[0., 0., 0.])]   # Sgr A*
for estado, masa in zip(estados_todos, masas_todos):
    x, y, z, vx, vy, vz = estado
    sistema_39.append(dict(m=G_canon * masa, r=[x, y, z], v=[vx, vy, vz]))

n_cuerpos_39 = len(sistema_39)
T_sim_39     = 100
N_pasos_39   = 5000
ts_39        = np.linspace(0, T_sim_39, N_pasos_39)

print(f"Sistema: {n_cuerpos_39} cuerpos  (Sgr A* + {N_estrellas} estrellas S)")
print(f"Integrando {T_sim_39} años con {N_pasos_39} pasos (dt = {T_sim_39/N_pasos_39:.3f} yr) ...")
rs_39, vs_39, _, _, const_39 = pc.ncuerpos_solucion(sistema_39, ts_39)
# rs_39.shape = (n_cuerpos_39, N_pasos_39, 3)
print(f"¡Integración completada!  rs_39.shape = {rs_39.shape}")

# ─── Conservación de energía ──────────────────────────────────────────────────
K_39, U_39 = const_39['K'], const_39['U']
E_39       = K_39 + U_39
err_E_39   = np.abs((E_39 - E_39[0]) / E_39[0]).max()
print(f"Error relativo máx. en energía ({N_estrellas} estrellas): {err_E_39:.2e}")

fig_E39, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(ts_39, (E_39 - E_39[0]) / np.abs(E_39[0]), color='tab:red')
axs[0].set(xlabel="Tiempo (años)", ylabel="(E − E₀) / |E₀|",
           title=f"Conservación de energía — {N_estrellas} estrellas S")
axs[0].grid(True)

axs[1].plot(ts_39, K_39, label='K', color='tab:orange')
axs[1].plot(ts_39, U_39, label='U', color='tab:blue')
axs[1].plot(ts_39, E_39, label='E=K+U', color='tab:green', lw=2)
axs[1].set(xlabel="Tiempo (años)", ylabel="Energía (mpc² yr⁻²)",
           title="Componentes de energía")
axs[1].legend(); axs[1].grid(True)
plt.tight_layout(); plt.show()

---
## Visualización interactiva con animación temporal

La figura siguiente muestra:
- **Líneas** (estáticas): trayectorias orbitales completas durante 100 años.
- **Marcadores** (animados): posición instantánea de cada cuerpo en el tiempo *t*.
- **Slider / botones ▶ ⏸**: control de la animación en 80 fotogramas.
- Disco negro en tonos azul; disco rojo en tonos rojo/naranja; Sgr A* en dorado.

In [ ]:
# ─── Paleta de colores por disco ─────────────────────────────────────────────
_pal_black = ['#4C9BE8','#5DADE2','#1A5276','#2E86C1','#AED6F1',
              '#1B4F72','#7FB3D3','#154360','#52BE80','#117A65',
              '#1E8449','#239B56','#27AE60','#2ECC71','#82E0AA',
              '#A9DFBF','#145A32','#196F3D','#1D8348']
_pal_red   = ['#E74C3C','#C0392B','#922B21','#CB4335','#F1948A',
              '#F5CBA7','#E59866','#CA6F1E','#D4AC0D','#B7950B',
              '#9A7D0A','#7D6608','#F0B27A','#E67E22','#DC7633',
              '#A04000','#7E5109','#B9770E','#FAD7A0','#F8C471']
_cnt_b, _cnt_r = 0, 0

def _color(idx):
    global _cnt_b, _cnt_r
    if idx == 0:
        return 'gold'
    disco = disco_todos[idx - 1]
    if disco == 'black':
        c = _pal_black[_cnt_b % len(_pal_black)]; _cnt_b += 1
    else:
        c = _pal_red[_cnt_r % len(_pal_red)];     _cnt_r += 1
    return c

nombres_39_full = ['Sgr A*'] + nombres_todos
colors_39       = [_color(i) for i in range(n_cuerpos_39)]

# ─── Trazas estáticas (trayectorias completas) ───────────────────────────────
# rs_39[i, :, :] = trayectoria del cuerpo i a lo largo del tiempo
static_traces = []
for i in range(n_cuerpos_39):
    label = nombres_39_full[i]
    static_traces.append(go.Scatter3d(
        x=rs_39[i, :, 0], y=rs_39[i, :, 1], z=rs_39[i, :, 2],
        mode='lines',
        line=dict(width=3 if i == 0 else 1, color=colors_39[i]),
        name=label, opacity=1.0 if i == 0 else 0.55,
    ))

# ─── Trazas animadas (marcador de posición instantánea) ───────────────────────
# rs_39[i, 0, :] = posición inicial del cuerpo i
marker_traces = []
for i in range(n_cuerpos_39):
    label = nombres_39_full[i]
    x0, y0, z0 = rs_39[i, 0, 0], rs_39[i, 0, 1], rs_39[i, 0, 2]
    marker_traces.append(go.Scatter3d(
        x=[x0], y=[y0], z=[z0],
        mode='markers',
        marker=dict(size=9 if i == 0 else 5, color=colors_39[i],
                    symbol='diamond' if i == 0 else 'circle',
                    line=dict(width=1, color='white')),
        name=label, showlegend=False,
        hovertemplate=(f'<b>{label}</b><br>'
                       'x=%{x:.3f} mpc<br>y=%{y:.3f} mpc<br>z=%{z:.3f} mpc<extra></extra>'),
    ))

# ─── Frames de animación (80 fotogramas) ─────────────────────────────────────
# En cada frame solo se actualizan los marcadores (traces n_cuerpos_39 .. 2*n_cuerpos_39-1)
N_frames   = 80
frame_idxs = np.linspace(0, N_pasos_39 - 1, N_frames, dtype=int)

frames = []
for fi, idx in enumerate(frame_idxs):
    # rs_39[i, idx, :] = posición del cuerpo i en el instante ts_39[idx]
    frame_data = [
        go.Scatter3d(x=[rs_39[i, idx, 0]],
                     y=[rs_39[i, idx, 1]],
                     z=[rs_39[i, idx, 2]])
        for i in range(n_cuerpos_39)
    ]
    frames.append(go.Frame(
        data=frame_data,
        traces=list(range(n_cuerpos_39, 2 * n_cuerpos_39)),
        name=str(fi),
    ))

# ─── Slider de tiempo ────────────────────────────────────────────────────────
slider_steps = [
    dict(args=[[str(fi)], dict(frame=dict(duration=80, redraw=True), mode='immediate')],
         label=f'{ts_39[idx]:.0f} yr', method='animate')
    for fi, idx in enumerate(frame_idxs)
]

# ─── Anotaciones de texto para estrellas destacadas ──────────────────────────
star_labels = []
for star, text in [('S2',   '★ S2'),
                   ('S62',  'S62 e=0.98'),
                   ('S55',  'S55'),
                   ('S175', 'S175 e≈0.999')]:
    if star in nombres_39_full:
        si = nombres_39_full.index(star)
        x0, y0, z0 = rs_39[si, 0, 0], rs_39[si, 0, 1], rs_39[si, 0, 2]
        star_labels.append(go.Scatter3d(
            x=[x0], y=[y0], z=[z0],
            mode='text',
            text=[text],
            textfont=dict(size=11, color='white'),
            showlegend=False, name=f'{star}_label',
        ))

# ─── Figura final ─────────────────────────────────────────────────────────────
fig_39 = go.Figure(
    data=static_traces + marker_traces + star_labels,
    frames=frames,
    layout=go.Layout(
        title=dict(text=f'Sgr A* + {N_estrellas} estrellas S — {T_sim_39} años',
                   font=dict(size=15)),
        scene=dict(xaxis_title='x (mpc)', yaxis_title='y (mpc)', zaxis_title='z (mpc)',
                   aspectmode='data'),
        legend=dict(font=dict(size=9), itemsizing='constant'),
        margin=dict(l=0, r=0, t=50, b=130),
        updatemenus=[dict(
            type='buttons', showactive=False, direction='left',
            x=0.05, y=-0.08, xanchor='left',
            buttons=[
                dict(label='▶ Play',
                     method='animate',
                     args=[None, dict(frame=dict(duration=80, redraw=True),
                                      fromcurrent=True)]),
                dict(label='⏸ Pausa',
                     method='animate',
                     args=[[None], dict(frame=dict(duration=0, redraw=False),
                                        mode='immediate')]),
            ],
        )],
        sliders=[dict(
            active=0, steps=slider_steps,
            x=0.05, len=0.9, xanchor='left',
            y=-0.14, yanchor='top',
            currentvalue=dict(prefix='Tiempo: ', font=dict(size=13)),
            pad=dict(t=40),
        )],
    )
)
fig_39.show()

---
## Análisis cuantitativo de perturbaciones — 39 estrellas

Se repite la comparación N-cuerpos vs. 2-cuerpos para el conjunto completo.
El cociente de fuerzas teórico es:

$$\frac{F_\text{estrellas}}{F_{\text{Sgr A*}}} \approx
\frac{N_{\rm est}\,\langle m \rangle}{4\,M_{\text{Sgr A*}}} \sim \mathcal{O}(10^{-5})$$

El gráfico de barras muestra la desviación relativa máxima $\max_t (\Delta r/|r|)$ por estrella.

In [ ]:
# ─── 2-cuerpos para las 39 estrellas ─────────────────────────────────────────
# rs_2b_i[1, :, :] = trayectoria kepleriana de la estrella i
print(f"Calculando {N_estrellas} trayectorias kepleranas (puede tardar ~2 min en Colab)...")
max_dr_rel = []

for i, (nombre, estado, masa) in enumerate(zip(nombres_todos, estados_todos, masas_todos)):
    sistema_2b = [
        dict(m=mu_SgrA,        r=[0., 0., 0.],     v=[0., 0., 0.]),
        dict(m=G_canon * masa,  r=list(estado[:3]), v=list(estado[3:])),
    ]
    rs_2b_i, _, _, _, _ = pc.ncuerpos_solucion(sistema_2b, ts_39)
    r_nb_i  = rs_39[i + 1, :, :]          # trayectoria N+1 cuerpos
    r_2b_i  = rs_2b_i[1, :, :]            # trayectoria kepleriana
    dr_rel  = (np.linalg.norm(r_nb_i - r_2b_i, axis=1) /
               (np.linalg.norm(r_nb_i, axis=1) + 1e-30)).max()
    max_dr_rel.append(dr_rel)
    if (i + 1) % 10 == 0:
        print(f"  {i+1}/{N_estrellas} completadas...")

print("¡Completado!")

# ─── Gráfico de barras ────────────────────────────────────────────────────────
bar_colors = ['steelblue' if d == 'black' else 'tomato' for d in disco_todos]

fig_pert, ax_pert = plt.subplots(figsize=(14, 5))
ax_pert.bar(nombres_todos, max_dr_rel, color=bar_colors)
ax_pert.set_yscale('log')
ax_pert.set(xlabel="Estrella",
            ylabel=r"Desviación relativa máx.  $\Delta r / |r|$",
            title="Perturbaciones N-cuerpos vs. Kepler puro (2-cuerpos)")
ax_pert.tick_params(axis='x', rotation=90)
ax_pert.grid(True, axis='y', alpha=0.5)

ratio_total = np.sum(masas_todos) / 4.3e6
ax_pert.axhline(ratio_total, ls='--', color='gray')
ax_pert.legend(handles=[
    mpatches.Patch(color='steelblue', label='Disco negro (inner cluster)'),
    mpatches.Patch(color='tomato',    label='Disco rojo  (outer cluster)'),
    plt.Line2D([0], [0], ls='--', color='gray',
               label=fr'$M_{{\rm est}}/M_{{\rm Sgr\,A*}} \approx {ratio_total:.1e}$'),
])
plt.tight_layout(); plt.show()

# ─── Resumen numérico ─────────────────────────────────────────────────────────
i_max = int(np.argmax(max_dr_rel)); i_min = int(np.argmin(max_dr_rel))
print(f"\nResumen de perturbaciones estrella–estrella:")
print(f"  M_total_estrellas / M_SgrA  = {ratio_total:.2e}")
print(f"  Desviación relativa media   = {np.mean(max_dr_rel):.2e}")
print(f"  Desviación máxima           = {max_dr_rel[i_max]:.2e}  ({nombres_todos[i_max]})")
print(f"  Desviación mínima           = {max_dr_rel[i_min]:.2e}  ({nombres_todos[i_min]})")
print(f"\n→ Perturbaciones de orden {np.mean(max_dr_rel):.0e}, "
      f"coherente con la relación de masas M_est/M_SgrA ~ {ratio_total:.0e}.")

---
## Conclusiones

### 1. Dinámica Newtoniana base
Las 39 estrellas del cúmulo S mantienen sus órbitas elípticas kepleranas durante los 100 años
simulados.  Las estrellas del disco negro (cúmulo interno, $a < 50$ mpc) completan múltiples
órbitas, mientras las del disco rojo (cúmulo externo) apenas avanzan una fracción de su período.
La estrella **S2** ($a = 5.03$ mpc, $e = 0.887$, $T \approx 16$ yr) completa ∼6 órbitas en la
simulación, y su trayectoria concuerda con los valores observacionales.

### 2. Perturbaciones de N-cuerpos
La masa total de las estrellas es $\sum m_i \approx 430\,M_\odot$, frente a
$M_{\rm Sgr\,A*} \approx 4.3 \times 10^6\,M_\odot$:

$$\frac{F_{\rm estrellas}}{F_{\rm Sgr\,A*}} \sim
\frac{M_{\rm est}}{M_{\rm Sgr\,A*}} \sim 10^{-4}$$

Numéricamente, la desviación relativa máxima acumulada tras 100 años oscila entre $10^{-6}$ y
$10^{-2}$ según la estrella.  Las más perturbadas son aquellas con períodos largos (más tiempo en
el cúmulo externo, donde las distancias inter-estelares son comparativamente menores respecto al
dominio de Sgr A*).

**En términos físicos:** bajo la dinámica Newtoniana pura, el centro galáctico está completamente
dominado por el agujero negro.  Las perturbaciones mutuas entre estrellas son despreciables en
escalas de décadas, pero pueden acumularse significativamente en escalas de $10^3$–$10^4$ años.